# Project 3: hoeveel stegen de prijzen in Suriname?

**De vraag.** Hoeveel stegen de prijzen in Suriname van 2010 tot 2025, en wat kunnen we niet zeggen over 2018 en 2019?

**De data.** De reeks van de Wereldbank met de inflatie van consumentenprijzen in Suriname, in procent per jaar (indicator FP.CPI.TOTL.ZG). Elke waarde is de verandering in de gemiddelde kosten van een pakket goederen en diensten ten opzichte van het jaar ervoor. De Wereldbank haalt de reeks uit de International Financial Statistics van het Internationaal Monetair Fonds en heeft hem voor het laatst bijgewerkt op 2026-10-08. De tabel is gedownload op 2026-11-06 en heeft geen waarde voor 2018 of 2019.

**Je team.** Vier rollen wisselen elke week: bouwer, datacontroleur, AI-controleur en presentator. Deze week zorgt de datacontroleur ervoor dat geen enkel getal op het prijzenblad afhangt van een jaar dat de reeks niet heeft.

**Voor wie het is.** Een buurtorganisatie in Suriname die ouderen met een vast pensioen ondersteunt. De organisatie krijgt een prijzenblad van ongeveer 200 woorden dat een vrijwilliger op een bijeenkomst kan voorlezen.

**Wat je vrijdag inlevert.** Dit notebook met elke CHECK geslaagd, één grafiek, het prijzenblad voor de buurtorganisatie, en jullie AI-logboek.

## Laad de data

Deze cel krijg je cadeau. Hij leest de tabel uit de cursusmap, of rechtstreeks bij de Wereldbank als het bestand er niet is. De jaren 2018 en 2019 staan in de tabel als rijen met een lege waarde.

In [ ]:
import json, urllib.request
import pandas as pd

URL = ("https://api.worldbank.org/v2/country/SUR/indicator/FP.CPI.TOTL.ZG"
       "?format=json&date=2010:2025&per_page=100")
try:
    inflation = pd.read_csv("https://santerre.ai/bigi-data/data/week03-inflation-2010-2025.csv")
except FileNotFoundError:
    raw = json.load(urllib.request.urlopen(URL))[1]
    inflation = pd.DataFrame(
        [{"year": int(r["date"]), "inflation_pct": r["value"]} for r in raw]
    ).sort_values("year").reset_index(drop=True)
inflation

## TODO 1: beschrijf de tabel

Druk het aantal rijen en het eerste en laatste jaar af. Sla in `missing_years` een lijst op van de jaren zonder waarde.

In [ ]:
# TODO 1
missing_years = None

In [ ]:
# CHECK 1
expected = inflation.loc[inflation["inflation_pct"].isna(), "year"].tolist()
assert missing_years is not None, "missing_years is nog leeg"
assert sorted(int(y) for y in missing_years) == expected
print("CHECK 1 geslaagd:", expected)

## TODO 2: maak een grafiek van de inflatie, met de gaten

Maak een staafgrafiek van de inflatie per jaar. Laat 2018 en 2019 leeg, en zet bij die jaren in de grafiek dat ze ontbreken. Geef de grafiek een titel, benoemde assen en de bron.

In [ ]:
# TODO 2

## TODO 3: van procent naar groeifactor

Voeg een kolom `factor` toe aan `inflation`: het getal waarmee de prijzen van elk jaar werden vermenigvuldigd, dat is 1 plus het percentage gedeeld door 100. De jaren 2018 en 2019 moeten leeg blijven.

In [ ]:
# TODO 3

In [ ]:
# CHECK 3
assert "factor" in inflation, "inflation heeft nog geen kolom factor"
p = inflation.set_index("year")["inflation_pct"]
f = inflation.set_index("year")["factor"]
assert ((f - (1 + p / 100)).abs().dropna() < 1e-9).all(), "een stijging van p procent vermenigvuldigt de prijzen met 1 + p/100"
assert f[p.isna()].isna().all(), "de ontbrekende jaren moeten leeg blijven, niet 0 of 1"
print("CHECK 3 geslaagd: de grootste factor is", round(f.max(), 4), "in", int(f.idxmax()))

## TODO 4: de twee stukken die de data dekt

De waarde voor 2011 vergelijkt de prijzen in 2011 met de prijzen in 2010. Bepaal welke jaren elk van de twee stijgingen hieronder gebruikt voordat je code schrijft.

Sla in `rise_2010_2017` op hoeveel keer zo hoog de prijzen in 2017 waren als in 2010, en in `rise_2019_2025` hoeveel keer zo hoog ze in 2025 waren als in 2019. Druk beide ook af als procentuele stijging.

In [ ]:
# TODO 4
rise_2010_2017 = None
rise_2019_2025 = None

In [ ]:
# CHECK 4
g = 1 + inflation.set_index("year")["inflation_pct"] / 100
assert rise_2010_2017 is not None and rise_2019_2025 is not None, "een van de twee stijgingen is nog leeg"
assert rise_2010_2017 < 100, "sla een veelvoud op (2 betekent dat de prijzen verdubbelden), geen percentage"
assert abs(rise_2010_2017 - g.loc[2011:2017].prod()) < 1e-6, "de waarde voor 2010 vergelijkt 2010 met 2009, dus die hoort niet bij de stijging vanaf 2010"
assert abs(rise_2019_2025 - g.loc[2020:2025].prod()) < 1e-6, "de stijging vanaf 2019 begint met de waarde voor 2020"
print("CHECK 4 geslaagd:", round(rise_2010_2017, 2), "keer en", round(rise_2019_2025, 2), "keer")

## TODO 5: wat de reeks niet kan zeggen

De stijging van 2010 tot 2025 heeft elke factor van 2011 tot 2025 nodig, ook die van 2018 en 2019. Sla die stijging op in `total_2010_2025`, zo berekend dat het antwoord ontbreekt (`prod(skipna=False)`).

Sla daarna in `at_least` op hoe groot die stijging minstens was als de prijzen in 2018 en 2019 niet daalden, als veelvoud van het prijsniveau van 2010. Schrijf de aanname als commentaar ernaast.

In [ ]:
# TODO 5
total_2010_2025 = None
at_least = None

In [ ]:
# CHECK 5
import math
g = 1 + inflation.set_index("year")["inflation_pct"] / 100
assert total_2010_2025 is not None, "total_2010_2025 is nog leeg: het moet een ontbrekende waarde zijn"
assert math.isnan(total_2010_2025), "dit totaal heeft 2018 en 2019 nodig, en die heeft de reeks niet"
assert at_least is not None, "at_least is nog leeg"
assert abs(at_least - g.loc[2011:2017].prod() * g.loc[2020:2025].prod()) < 1e-6
print("CHECK 5 geslaagd: het totaal is onbekend, en het is minstens", round(at_least, 2), "keer als de prijzen in 2018 en 2019 niet daalden")

## TODO 6: de AI-controle

Plak de tabel in je AI-assistent en vraag: *"Hoeveel zijn de consumentenprijzen in Suriname volgens deze tabel met jaarlijkse inflatie in totaal gestegen van 2010 tot 2025?"* Schrijf het antwoord woord voor woord in jullie AI-logboek.

Bereken daarna drie antwoorden die een stille fout zou opleveren, elk als procentuele stijging van 2010 tot 2025, en kijk of het antwoord van de AI dicht bij een ervan ligt.

- `if_added` telt de percentages voor 2011 tot en met 2025 op.
- `if_skipped` vermenigvuldigt de factoren voor 2011 tot en met 2025 en slaat 2018 en 2019 over.
- `if_filled` vermenigvuldigt dezelfde factoren nadat 2018 en 2019 zijn ingevuld met een rechte lijn tussen 2017 en 2020 (`interpolate()`).

Vraag de assistent ten slotte zonder de tabel: *"Hoe hoog was de inflatie in Suriname in 2018 en in 2019?"* Noteer de getallen en de bron die de assistent noemt. De reeks van de Wereldbank heeft voor die jaren geen waarde, dus een getal zonder een bron die je zelf hebt geopend, komt niet op je prijzenblad.

In [ ]:
# TODO 6
if_added = None
if_skipped = None
if_filled = None

In [ ]:
# CHECK 6
p = inflation.set_index("year")["inflation_pct"]
g = 1 + p / 100
assert None not in (if_added, if_skipped, if_filled), "een van de drie is nog leeg"
assert abs(if_added - p.loc[2011:2025].sum()) < 1e-6, "if_added: tel de percentages voor 2011 tot en met 2025 op"
assert abs(if_skipped - (g.loc[2011:2025].prod() - 1) * 100) < 1e-6, "if_skipped: een procentuele stijging, met 2018 en 2019 overgeslagen"
assert abs(if_filled - ((1 + p.interpolate() / 100).loc[2011:2025].prod() - 1) * 100) < 1e-6, "if_filled: vul eerst in met interpolate()"
print("CHECK 6 geslaagd:", round(if_added, 1), round(if_skipped, 1), round(if_filled, 1))

## TODO 7: het prijzenblad voor de buurtorganisatie

Schrijf in een tekstcel hieronder ongeveer 200 woorden die een vrijwilliger kan voorlezen aan ouderen met een vast pensioen. Geef de stijging voor elk stuk dat de data dekt, en zeg wat een bedrag dat sinds 2019 gelijk is gebleven in 2025 nog koopt, als deel van wat het in 2019 kocht. Zeg duidelijk dat deze reeks het totaal van 2010 tot 2025 niet kan geven, en waarom. Geef het laagste dat het totaal kan zijn, met de aanname erachter, en noem de bron. Gebruik gewone woorden en geen code.